In [ ]:
import scanpy as sc
import torch
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy
import pandas as pd
import decoupler as dc
import anndata
import os

sc.set_figure_params(figsize=(4, 4))
date = "DATE"

In [ ]:
base_path = "/home/EOCRC_atlas"

In [ ]:
# load in Tier2 annotated adata and remove program variables 
adata = sc.read_h5ad(os.path.join(base_path, "data/pelka_full.h5ad"))
print(adata.var_names[-206:])
adata = adata[:, :-204].copy()
print(adata.var_names[-4:])

In [ ]:
# sample list from inferCNV outputs 
samples = os.listdir(os.path.join(base_path, 'results/PELKA_inferCNV/outputs_for_VM/outputs'))

In [ ]:
# load in inferCNV results 
dfs = []

for d in samples: 
    df = pd.read_csv(os.path.join(base_path, f'results/PELKA_inferCNV/outputs_for_VM/outputs/{d}/infercnv/HMM_CNV_predictions.HMMi6.leiden.hmm_mode-subclusters.Pnorm_0.5.pred_cnv_genes.dat'), sep='\t')
        
    # remove the non-epithelial data
    df = df[~df['cell_group_name'].str.contains('Non-epithelial', case=False, na=False)]
    
    # add sample ID to the df 
    df['cell_group_name'] = d+'_'+df['cell_group_name']

    # look up sample age information 
    sample_type = np.unique(adata.obs['SpecimenType'][adata.obs['PatientTypeID']==d])
    msi = np.unique(adata.obs['MMR-IHC'][adata.obs['PatientTypeID']==d])

    df['sample_type']=sample_type[0]
    df['msi']=msi[0]

    dfs.append(df)

In [ ]:
# Create an empty list to store the intermediate results
chunk_size = 10  # number of DataFrames per chunk
chunks = []
for i in range(0, len(dfs), chunk_size):
    chunk = pd.concat(dfs[i:i+chunk_size])
    chunks.append(chunk)
df_all = pd.concat(chunks)

In [ ]:
# rename normal samples 
df_all.loc[df_all['sample_type'] == "N", 'msi'] = "Normal"

In [ ]:
chrom = df_all[['gene', 'chr']].drop_duplicates()
chrom.index = chrom['gene']

In [ ]:
# load in chr/gene order 
chr_order = pd.read_csv(os.path.join(base_path, 'docs/Trinity_CTAT_cnv_hg38_gencode_v27.txt'), sep='\t', usecols=[0, 1, 2, 3], header=None) #update
chr_order.columns=['gene', 'chr', 'start', 'end']

In [ ]:
# Make a cell_group_name x gene/chr matrix 
df_wide = df_all.pivot(index='cell_group_name', columns='gene', values='state')
genes_order = df_all['gene'].drop_duplicates()
df_wide = df_wide[genes_order]
df_wide.fillna(3, inplace=True) # replacing with 3 becuase in the HMM values, a value of 3 is "white" - do deletion or duplication 
df_wide
metadata = df_all[['cell_group_name', 'sample_type', 'msi']].drop_duplicates()
df_wide = df_wide.merge(metadata, left_index=True, right_on='cell_group_name', how='left')
df_wide.index = df_wide['cell_group_name']

In [ ]:
gene_exists = pd.Index(chr_order['gene']).intersection(df_wide.columns)
df_wide_sorted = df_wide[gene_exists]

In [ ]:
# create vector with the CNV cluster number 
cnv_clusters = df_wide_sorted.index
s_label = cnv_clusters.str.split('_')

In [ ]:
df_wide_orig = df_wide
df_wide_sorted_orig = df_wide_sorted

# Create Pelka MSS + normal only file 

In [ ]:
df_wide = df_wide_orig[(df_wide_orig['msi'] == 'MSS') | (df_wide_orig['msi'] == 'Normal')]
df_wide_sorted = df_wide_sorted_orig[(df_wide_orig['msi'] == 'MSS') | (df_wide_orig['msi'] == 'Normal')]

In [ ]:
import pickle
with open(os.path.join(base_path, 'results/2025-10-01_YOCRC_inferCNV/MSS_dataframes_PELKA_06-28-26.pkl'), "wb") as f:
    pickle.dump({
        "df_wide": df_wide,
        "df_wide_sorted": df_wide_sorted
    }, f)

# comparing our MSS CNVs to Pelka et al

In [ ]:
# load our data 
with open(os.path.join(base_path, 'results/2025-10-01_YOCRC_inferCNV/MSS_dataframes_YOCRC.pkl'), "rb") as f:
    data = pickle.load(f)

df_wide_yocrc = data["df_wide"]
df_wide_sorted_yocrc = data["df_wide_sorted"]
df_wide_yocrc['sample_type']='yocrc'
df_wide_yocrc['msi']=df_wide_yocrc['MSI']
df_wide_yocrc = df_wide_yocrc.drop(columns=['Decade', 'Cohort', 'MSI', 'Side'])

In [ ]:
# load pelka data 
with open(os.path.join(base_path, 'results/2025-10-01_YOCRC_inferCNV/MSS_dataframes_PELKA_06-28-26.pkl'), "rb") as f:
    data = pickle.load(f)

df_wide_pelka = data["df_wide"]
df_wide_sorted_pelka = data["df_wide_sorted"]

In [ ]:
shared_cols = df_wide_yocrc.columns.intersection(df_wide_pelka.columns)
print("Number of shared columns:", len(shared_cols))

In [ ]:
print(df_wide_pelka.shape)
print(df_wide_yocrc.shape)

In [ ]:
df_yocrc_shared = df_wide_yocrc[shared_cols]
df_pelka_shared = df_wide_pelka[shared_cols]
df_wide_merged = pd.concat([df_yocrc_shared, df_pelka_shared], axis=0, ignore_index=True)

In [ ]:
shared_cols = df_wide_sorted_yocrc.columns.intersection(df_wide_sorted_pelka.columns)
print("Number of shared columns:", len(shared_cols))

In [ ]:
df_yocrc_shared = df_wide_sorted_yocrc[shared_cols]
df_pelka_shared = df_wide_sorted_pelka[shared_cols]
df_wide_sorted_merged = pd.concat([df_yocrc_shared, df_pelka_shared], axis=0, ignore_index=False)

In [ ]:
import seaborn as sns
import matplotlib.colors as mcolors
from scipy.spatial import distance
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster

row_dist = distance.pdist(df_wide_sorted_merged, metric = 'correlation')
row_linkage = hierarchy.linkage(row_dist, method='ward', metric='correlation')

clusters = fcluster(row_linkage, t=2.5, criterion='distance')  
clusters
df_clusters = pd.DataFrame({'cluster': clusters}, index=df_wide_sorted_merged.index)

hex_colors = [
           "#0B6E4F", "#84B8F7", "#A1D47E", "#E9B2A6", "#C6D4E2", "#F9D79C", 
           "#A9D0F5", "#77A1D4", "#D1A1F1", "#8C1A61", "#C0F2A1", "#E19F67", "#2AAB62", "#F1A943", 
           "#BB74D2", "#65D7D2", "#A76F8B", "#C9A08A", "#63B2B3", "#F4A7BC", "#5A6261", 
           "#FFD16C", "#70B4AE", "#34A8A1", "#E4C697", "#B9C6D5", "#D4A358", "#4F8D7C", "#BEF1B6"]
lut = dict(zip(df_clusters['cluster'].unique(), hex_colors[18:len(hex_colors)]))

cluster_colors = df_clusters['cluster'].map(lut)

df_wide_merged.index = df_wide_sorted_merged.index
cohort_colors = df_wide_merged['sample_type'].map({'T': '#56B4E9', 'N': '#E69F00', 'yocrc': '#F55D4F'})

row_colors = pd.DataFrame({'sample_type': cohort_colors, 'Cluster': cluster_colors})

col_annotation = chrom['chr']
col_colors = col_annotation.map({'chr1': 'Crimson', 
                                 'chr2': 'SlateBlue', 
                                 'chr3': 'ForestGreen', 
                                 'chr4': 'DodgerBlue', 
                                 'chr5': 'MediumPurple', 
                                 'chr6': 'Tomato', 
                                 'chr7': 'Goldenrod', 
                                 'chr8': 'Coral', 
                                 'chr9': 'OliveDrab', 
                                 'chr10': 'DarkOrange', 
                                 'chr11': 'MediumSeaGreen', 
                                 'chr12': 'SteelBlue', 
                                 'chr13': 'RoyalBlue', 
                                 'chr14': 'Indigo', 
                                 'chr15': 'LimeGreen', 
                                 'chr16': 'PapayaWhip', 
                                 'chr17': 'Turquoise', 
                                 'chr18': 'Chocolate', 
                                 'chr19': 'FireBrick', 
                                 'chr20': 'HotPink', 
                                 'chr21': 'LavenderBlush', 
                                 'chr22': 'SeaGreen', 
                                 'chrM': 'SaddleBrown', 
                                 'chrX': 'MediumVioletRed', 
                                 'chrY': 'SkyBlue'})

cmap = mcolors.LinearSegmentedColormap.from_list("blue_white_red", ["blue", "white", "red"])
norm = mcolors.TwoSlopeNorm(vmin=df_wide_sorted_merged.min().min(), vcenter=3, vmax=df_wide_sorted_merged.max().max())

ax = sns.clustermap(df_wide_sorted_merged, 
                    row_linkage=row_linkage, 
                    #row_cluster=True, 
                    col_cluster=False,
                    row_colors=row_colors, 
                    col_colors=col_colors,
                    cmap=cmap, 
                    norm=norm, 
                    figsize=(25,df_wide.shape[0]/10))

ax.ax_heatmap.set_xticklabels([])  # Remove x-axis tick labels
ax.ax_heatmap.set_yticklabels([])
ax.ax_heatmap.set_xlabel("Gene/Chromosome")
ax.ax_heatmap.set_ylabel("SampleID-InferCNV-Cluster")
ax.ax_heatmap.grid(False)

ax.savefig(os.path.join(base_path, 'results/EOCRC_inferCNV/allSamples_HMM_clustermap_MSS_yocrc_and_pelka.pdf'), dpi=600)
plt.show()

In [ ]:
row_order = ax.dendrogram_row.reordered_ind
sorted_sample_ids = df_wide_sorted_merged.index[row_order]

df_cluster_export = pd.DataFrame({
    "sample_id": sorted_sample_ids,
    "cluster": clusters[row_order]
})

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df_cluster_export.head()

In [ ]:
df_wide_sorted_merged.head()

In [ ]:
# calculate hmm burden 
hmm_burden = np.abs(df_wide_sorted_merged - 3).sum(axis=1)

df_hmm = pd.DataFrame({
    'hmm_burden': hmm_burden
})

cluster_df = df_cluster_export
cluster_df = cluster_df.set_index('sample_id')

final_df = df_hmm.join(cluster_df['cluster'].astype(str), how='inner')
final_df['meta_cluster'] = 'Cluster ' + final_df['cluster']

# make box plot 
plt.figure(figsize=(3, 3), dpi=300)

cluster_order = ['Cluster 1', 'Cluster 5', 'Cluster 3', 'Cluster 2', 'Cluster 4']

sns.boxplot(
    data=final_df,
    x='meta_cluster',
    y='hmm_burden',
    order=cluster_order,
    width=0.4,
    fliersize=0,
    boxprops=dict(facecolor='lightsteelblue'),
    showcaps=True
)

sns.stripplot(
    data=final_df,
    x='meta_cluster',
    y='hmm_burden',
    order=cluster_order,
    color='black',
    size=2,
    alpha=0.4,
    jitter=0.15
)

plt.title('CNA Burden Across Tumor and Normal Epithelium', fontsize=7, fontweight='bold', pad=12)
plt.xlabel('Meta-cluster', fontsize=7)
plt.ylabel('CNA Burden\nsum|HMM state − 3|', fontsize=7)
plt.xticks(fontsize=5)
plt.yticks(fontsize=5)
sns.despine()
plt.tight_layout()

plt.savefig(os.path.join(base_path, 'results/EOCRC_inferCNV/HMM_Burden_Pelak_Porter_6-29-26.pdf'), format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
cna_burden_per_subset = (df_wide_sorted_merged - 3).abs().sum(axis=1)

# Or binary: does any gene deviate from neutral
any_nonneutral = (df_wide_sorted_merged != 3).any(axis=1)

# Fraction of subsets with any non-neutral state per sample
# (assuming you have a sample ID column or index)
subset_summary = pd.DataFrame({
    'cna_burden': hmm_burden,
    'any_nonneutral': any_nonneutral,
    'sample_id': df_wide_sorted_merged.index  # however samples are labeled
})

subset_summary['sample'] = subset_summary['sample_id'].str.extract(r'^(.+?)_all_')
subset_summary = subset_summary.merge(df_cluster_export, left_on='sample_id', right_on="sample_id", how='left')
subset_summary

In [ ]:
subset_summary.groupby('cluster')['cna_burden'].describe()

In [ ]:
# wilcox test between pelka normal and our low CNA meta cluster 
from scipy.stats import mannwhitneyu
c1 = subset_summary[subset_summary['cluster'] == 1]['cna_burden']
c5 = subset_summary[subset_summary['cluster'] == 5]['cna_burden']
mannwhitneyu(c5, c1, alternative='greater')